<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/Phase52.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 · Phase 52 — authorized real-source normalization and private evidence freeze

Phase 51 established that the selected 258-person neuroimaging derivative uses a **wide longitudinal schema** with:

- baseline and M12 encoded columns;
- left/right encoding;
- bilateral hippocampal baseline→M12 fields;
- entorhinal, middle-temporal, ventricular, and ICV-related features.

Phase 52 does **not** run the Transformation Fidelity Layer yet. Fidelity evaluation requires a source→target transformation. Instead, this phase creates the immutable real-source reference against which later FHIR/OMOP transformations can be judged.

### Phase 52 objectives

1. Obtain an explicit local investigator authorization attestation.
2. Re-locate the exact Phase 51 source by its non-identifying token.
3. Freeze a **private source manifest** containing the exact path, source SHA-256, original column names, and local authorization metadata.
4. Normalize the wide source into a private longitudinal evidence table with deterministic private person keys.
5. Preserve missing semantics as missing: no invented units, DICOM identifiers, model versions, provenance, or diagnosis.
6. Publish only aggregate/non-identifying evidence.

### Important boundary

A valid authorization attestation allows local analysis. It does not establish clinical validity, official OMOP/MI-CDM conformance, or permission to publish participant-level source data.


In [1]:
# CODE CELL 1/9 — Drive, constants, helpers

import os, re, csv, json, gzip, hashlib, tempfile, shutil, zipfile, time, secrets, getpass
import datetime as dt
from pathlib import Path
from collections import defaultdict, Counter

PROJECT='ICHI2027_neuroimaging_fhir_omop_fidelity'
PHASE='52_authorized_real_source_normalization_and_private_evidence_freeze'
EXPECTED_P51_SHAREABLE='31ad86c5b95cf4f6ff46f69376e75d120ec809fdcde8bd8f2ba3f4d608e257ce'
SELECTED_SOURCE_TOKEN='069caf1537e082c6'

SUPPORTED={'.csv','.tsv','.txt','.gz','.parquet','.xlsx','.xls'}
SCAN_LIMIT_SECONDS=int(os.environ.get('PHASE52_SCAN_LIMIT_SECONDS','150'))
MAX_FS_ENTRIES=int(os.environ.get('PHASE52_MAX_FS_ENTRIES','30000'))
MAX_SCAN_DEPTH=int(os.environ.get('PHASE52_MAX_SCAN_DEPTH','6'))

def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(2*1024*1024),b''):
            h.update(b)
    return h.hexdigest()

def token_for_path(p):
    return hashlib.sha256(str(Path(p).resolve()).encode('utf-8')).hexdigest()[:16]

def write_csv(path,rows,fieldnames=None):
    rows=list(rows)
    if fieldnames is None:
        if not rows: raise ValueError('fieldnames required for empty CSV')
        fieldnames=list(rows[0].keys())
    with Path(path).open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=fieldnames)
        w.writeheader(); w.writerows(rows)

if os.environ.get('PHASE52_LOCAL_DRIVE_ROOT'):
    MYDRIVE=Path(os.environ['PHASE52_LOCAL_DRIVE_ROOT']).expanduser().resolve()
    ROOTS=[MYDRIVE]
else:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    MYDRIVE=Path('/content/drive/MyDrive')
    assert MYDRIVE.is_dir()
    ROOTS=[MYDRIVE]
    sd=Path('/content/drive/Shareddrives')
    if sd.is_dir(): ROOTS.append(sd)

hint=os.environ.get('PHASE52_DRIVE_FOLDER_HINT','').strip()
if hint:
    hp=Path(hint).expanduser()
    if not hp.is_absolute(): hp=MYDRIVE/hp
    if not hp.is_dir(): raise FileNotFoundError(hp)
    ROOTS=[hp.resolve()]

WORK=Path(tempfile.mkdtemp(prefix='ichi2027_phase52_'))
STAGE=WORK/'public_outputs'; STAGE.mkdir()
OUT_BASE=Path(os.environ.get('PHASE52_OUTPUT_DIR',str(MYDRIVE/'ICHI2027'/'Phase52')))
OUT=OUT_BASE/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')

PRIVATE_ROOT=MYDRIVE/'ICHI2027'/'_PRIVATE_REAL_MRI'
PRIVATE_ROOT.mkdir(parents=True,exist_ok=True)
PRIVATE_RUN=PRIVATE_ROOT/'Phase52'/dt.datetime.now(dt.timezone.utc).strftime('run_%Y%m%dT%H%M%S_%fZ')
PRIVATE_RUN.mkdir(parents=True,exist_ok=False)

print('Public Phase52 output:',OUT)
print('Private real-source freeze workspace prepared.')


Mounted at /content/drive
Public Phase52 output: /content/drive/MyDrive/ICHI2027/Phase52/run_20261001T051828_306695Z
Private real-source freeze workspace prepared.


In [2]:
# CODE CELL 2/9 — Verify exact Phase51 evidence chain

def name_matches(actual,stem,ext):
    return re.fullmatch(re.escape(stem)+r'(?:-\d+|\s*\(\d+\))?'+re.escape(ext),actual,re.I) is not None

hits=[]
for root in ROOTS:
    for here,dirs,files in os.walk(root,topdown=True,followlinks=False):
        dirs[:]=[d for d in dirs if d not in {'.git','node_modules','.ipynb_checkpoints'} and not d.startswith('.Trash')]
        for fn in files:
            if name_matches(fn,'ICHI2027_Phase51_SHAREABLE','.json'):
                p=Path(here)/fn
                try:
                    if sha256(p)==EXPECTED_P51_SHAREABLE: hits.append(p)
                except OSError: pass

if not hits:
    raise FileNotFoundError('Verified Phase51 SHAREABLE.json was not found in Drive.')

P51=sorted(hits,key=str)[0]
p51=json.loads(P51.read_text('utf-8'))

assert p51['phase']=='51_deep_wide_schema_audit_longitudinal_neuroimaging'
assert p51['selected_source_token']==SELECTED_SOURCE_TOKEN
assert p51['phase50_schema_reinterpreted_by_deep_wide_parser'] is True
assert p51['header_summary']['wide_bilateral_hippocampus_baseline_m12_established'] is True
assert p51['header_summary']['baseline_encoded_columns']>0
assert p51['header_summary']['m12_encoded_columns']>0
assert p51['real_authorized_mri_processed_by_proposed_method'] is False

print('PASS: Phase51 exact evidence chain verified.')
print('Wide bilateral hippocampus baseline/M12 schema:',p51['header_summary']['wide_bilateral_hippocampus_baseline_m12_established'])


PASS: Phase51 exact evidence chain verified.
Wide bilateral hippocampus baseline/M12 schema: True


In [3]:
# CODE CELL 3/9 — Explicit private authorization attestation

ATTEST=PRIVATE_ROOT/'AUTHORIZED_SOURCE_ATTESTATION.json'

def yes(prompt):
    ans=input(prompt+' [yes/no]: ').strip().lower()
    return ans in {'yes','y'}

existing_valid=False
att={}
if ATTEST.exists():
    try:
        att=json.loads(ATTEST.read_text('utf-8'))
        day=dt.date.fromisoformat(str(att['attestation_date_utc']))
        existing_valid=(
          day<=dt.datetime.now(dt.timezone.utc).date()
          and att.get('authorized_access') is True
          and att.get('authorized_for_local_research_analysis') is True
          and att.get('data_use_terms_reviewed') is True
          and bool(str(att.get('attested_by','')).strip())
          and bool(str(att.get('source_description','')).strip())
        )
    except Exception:
        existing_valid=False

if existing_valid:
    print('Existing private authorization attestation is valid.')
else:
    print('A valid local authorization attestation is required before row-level source processing.')
    print('Answer YES only if each statement is true for the selected neuroimaging derivative source.')

    authorized_access=yes('I currently have authorized access to this source dataset')
    local_analysis=yes('The data-use terms allow me to analyze this dataset locally for this research')
    terms_reviewed=yes('I have reviewed the applicable data-use/access terms')
    public_restriction=yes('I understand participant-level source data must not be placed in public outputs')

    attested_by=input('Enter your name or initials for the PRIVATE attestation: ').strip()
    source_description=input('Enter a short PRIVATE source description (for example, authorized ADNI-derived FreeSurfer table): ').strip()

    att={
      'source_family':'authorized_neuroimaging_derivative',
      'authorized_access':authorized_access,
      'authorized_for_local_research_analysis':local_analysis,
      'data_use_terms_reviewed':terms_reviewed,
      'participant_level_public_release_prohibited_or_restricted':public_restriction,
      'attestation_date_utc':dt.datetime.now(dt.timezone.utc).date().isoformat(),
      'attested_by':attested_by,
      'source_description':source_description,
      'notes':'Local investigator attestation. This does not replace the governing data-use agreement.'
    }
    ATTEST.write_text(json.dumps(att,indent=2)+'\n',encoding='utf-8')

day=dt.date.fromisoformat(str(att['attestation_date_utc']))
attestation_valid=(
  day<=dt.datetime.now(dt.timezone.utc).date()
  and att.get('authorized_access') is True
  and att.get('authorized_for_local_research_analysis') is True
  and att.get('data_use_terms_reviewed') is True
  and att.get('participant_level_public_release_prohibited_or_restricted') is True
  and bool(str(att.get('attested_by','')).strip())
  and bool(str(att.get('source_description','')).strip())
)

if not attestation_valid:
    raise RuntimeError(
      'Authorization attestation is incomplete or contains a NO answer. '
      'Phase52 stops before reading participant-level source rows.'
    )

print('PASS: private local authorization attestation validated.')


A valid local authorization attestation is required before row-level source processing.
Answer YES only if each statement is true for the selected neuroimaging derivative source.
I currently have authorized access to this source dataset [yes/no]: YES
The data-use terms allow me to analyze this dataset locally for this research [yes/no]: YES
I have reviewed the applicable data-use/access terms [yes/no]: YES
I understand participant-level source data must not be placed in public outputs [yes/no]: YES
Enter your name or initials for the PRIVATE attestation: YES
Enter a short PRIVATE source description (for example, authorized ADNI-derived FreeSurfer table): Authorized ADNI-derived FreeSurfer longitudinal neuroimaging derivative table
PASS: private local authorization attestation validated.


In [4]:
# CODE CELL 4/9 — Re-locate exact source by token using bounded scan

PATH_HINTS=('adni','mri','neuro','freesurfer','free_surfer','ucsf','trace','roi','volume','hippoc','entorh')
priority_terms=('adni','mri','neuro','research','data','trace','freesurfer','project','ichi')

search_starts=[]
for root in ROOTS:
    try: top=[p for p in root.iterdir() if p.is_dir()]
    except Exception: top=[]
    search_starts.extend([p for p in top if any(t in p.name.lower() for t in priority_terms)])
    search_starts.append(root)

start=time.monotonic()
entries=0
selected_path=None
seen=set()
stack=[(Path(p),0) for p in reversed(search_starts)]

while stack:
    if time.monotonic()-start>SCAN_LIMIT_SECONDS or entries>=MAX_FS_ENTRIES: break
    here,depth=stack.pop()
    try:key=str(here.resolve())
    except:key=str(here)
    if key in seen: continue
    seen.add(key)

    low=str(here).lower()
    if re.search(r'[/\\]ichi2027[/\\]phase\d+',low): continue
    if any(x in low for x in ['/.trash','\\.trash','/.git','\\.git','node_modules','.ipynb_checkpoints']): continue

    try:
        with os.scandir(here) as it:
            for ent in it:
                entries+=1
                if time.monotonic()-start>SCAN_LIMIT_SECONDS or entries>=MAX_FS_ENTRIES: break
                try:
                    if ent.is_dir(follow_symlinks=False):
                        if depth<MAX_SCAN_DEPTH: stack.append((Path(ent.path),depth+1))
                        continue
                    if not ent.is_file(follow_symlinks=False): continue
                except OSError:
                    continue

                p=Path(ent.path)
                if p.suffix.lower() not in SUPPORTED: continue
                plow=(p.name+' '+str(p.parent)).lower()
                if not any(h in plow for h in PATH_HINTS): continue
                if token_for_path(p)==SELECTED_SOURCE_TOKEN:
                    selected_path=p
                    break
        if selected_path is not None: break
    except (PermissionError,OSError):
        continue

if selected_path is None:
    raise FileNotFoundError(
      'Could not re-locate the selected Phase51 source within the bounded scan. '
      'Set PHASE52_DRIVE_FOLDER_HINT to its parent research-data folder and rerun.'
    )

print('PASS: exact selected source re-located by token.')
print('Scan entries:',entries,'seconds:',round(time.monotonic()-start,2))


PASS: exact selected source re-located by token.
Scan entries: 2477 seconds: 0.96


In [5]:
# CODE CELL 5/9 — Load source privately and map wide longitudinal columns

import pandas as pd
import numpy as np

def load_table(p):
    suf=p.suffix.lower()
    if suf=='.gz': return pd.read_csv(p,compression='gzip',low_memory=False)
    if suf=='.csv': return pd.read_csv(p,low_memory=False)
    if suf=='.tsv': return pd.read_csv(p,sep='\t',low_memory=False)
    if suf=='.txt':
        try:return pd.read_csv(p,sep='\t',low_memory=False)
        except Exception:return pd.read_csv(p,low_memory=False)
    if suf=='.parquet': return pd.read_parquet(p)
    if suf in {'.xlsx','.xls'}: return pd.read_excel(p)
    raise ValueError(suf)

def norm(c):
    s=str(c).strip().lower().replace('³','3').replace('^3','3')
    return re.sub(r'[^a-z0-9]+','_',s).strip('_')

def anatomy(n):
    if 'hippoc' in n:return 'hippocampus'
    if 'entorh' in n:return 'entorhinal'
    if 'middle_temporal' in n or 'middletemp' in n or 'midtemporal' in n or 'midtemp' in n:return 'middle_temporal'
    if 'ventric' in n:return 'ventricle'
    if re.search(r'(^|_)icv($|_)',n) or 'intracranial' in n or 'etiv' in n:return 'icv'
    return ''

def laterality(n):
    if re.search(r'(^|_)(left|lh)($|_)',n) or n.startswith('lh_') or '_lh_' in n:return 'L'
    if re.search(r'(^|_)(right|rh)($|_)',n) or n.startswith('rh_') or '_rh_' in n:return 'R'
    if re.search(r'(^|_)l(hipp|entorh|middle|vent)',n):return 'L'
    if re.search(r'(^|_)r(hipp|entorh|middle|vent)',n):return 'R'
    return ''

def timepoint(n):
    if any(re.search(p,n) for p in [
        r'(^|_)(bl|baseline|m0|m00|month0|month_0|v0|visit0|visit_0)($|_)',
        r'(^|_)(screening_baseline)($|_)']):
        return 'BASELINE'
    if any(re.search(p,n) for p in [
        r'(^|_)(m12|month12|month_12|12m|12_month|12months|v12|visit12|visit_12|yr1|year1|year_1)($|_)']):
        return 'M12'
    return ''

def explicit_unit(n):
    if re.search(r'(^|_)mm3($|_)',n) or 'cubic_mm' in n:return 'mm3'
    if re.search(r'(^|_)cm3($|_)',n) or 'cubic_cm' in n:return 'cm3'
    return ''

df=load_table(selected_path)
cols=list(df.columns)

person_candidates=[
    c for c in cols
    if re.search(r'(^|_)(rid|ptid|subject|subject_id|participant|participant_id|person_id|patient_id)($|_)',norm(c))
]
if not person_candidates:
    raise RuntimeError('No stable person-key column could be identified from the selected source.')
person_col=person_candidates[0]

mapped=[]
for c in cols:
    n=norm(c)
    a=anatomy(n); tp=timepoint(n); side=laterality(n)
    if not a or not tp: continue
    mapped.append({
      'private_original_column':c,
      'column_token':hashlib.sha256(str(c).encode('utf-8')).hexdigest()[:16],
      'anatomy':a,
      'laterality':side,
      'timepoint':tp,
      'explicit_unit':explicit_unit(n)
    })

required=[
 ('hippocampus','L','BASELINE'),('hippocampus','R','BASELINE'),
 ('hippocampus','L','M12'),('hippocampus','R','M12')
]
available={(r['anatomy'],r['laterality'],r['timepoint']) for r in mapped}
if not set(required)<=available:
    raise RuntimeError('Required bilateral hippocampus baseline/M12 wide fields are no longer reproducible from the selected source.')

print('PASS: private source loaded and longitudinal wide schema reproduced.')
print('Rows:',len(df),'unique persons:',df[person_col].dropna().astype(str).nunique())
print('Mapped longitudinal feature columns:',len(mapped))


PASS: private source loaded and longitudinal wide schema reproduced.
Rows: 258 unique persons: 258
Mapped longitudinal feature columns: 16


In [6]:
# CODE CELL 6/9 — Freeze private source manifest and normalize to private long evidence

# Persistent private salt; never export.
SALT_FILE=PRIVATE_ROOT/'SOURCE_KEY_SALT.bin'
if SALT_FILE.exists():
    salt=SALT_FILE.read_bytes()
else:
    salt=secrets.token_bytes(32)
    SALT_FILE.write_bytes(salt)

def private_person_key(v):
    h=hashlib.sha256()
    h.update(salt)
    h.update(str(v).encode('utf-8'))
    return h.hexdigest()

source_sha=sha256(selected_path)
source_stat=selected_path.stat()

private_manifest={
 'project':PROJECT,
 'phase':PHASE,
 'source_file_token':SELECTED_SOURCE_TOKEN,
 'exact_private_path':str(selected_path),
 'source_sha256':source_sha,
 'source_bytes':source_stat.st_size,
 'source_rows':len(df),
 'source_columns':len(cols),
 'person_key_column':person_col,
 'original_columns':cols,
 'mapped_longitudinal_columns':mapped,
 'authorization_attestation':att,
 'created_utc':dt.datetime.now(dt.timezone.utc).isoformat()
}
PRIVATE_MANIFEST=PRIVATE_RUN/'ICHI2027_Phase52_PRIVATE_SOURCE_MANIFEST.json'
PRIVATE_MANIFEST.write_text(json.dumps(private_manifest,indent=2,default=str)+'\n',encoding='utf-8')

records=[]
for r in mapped:
    c=r['private_original_column']
    vals=df[[person_col,c]].copy()
    vals=vals[vals[c].notna()]
    for pid,val in vals.itertuples(index=False,name=None):
        try:
            numeric=float(val)
        except Exception:
            continue
        records.append({
          'person_key_private':private_person_key(pid),
          'source_file_token':SELECTED_SOURCE_TOKEN,
          'source_column_token':r['column_token'],
          'anatomy':r['anatomy'],
          'laterality':r['laterality'],
          'timepoint':r['timepoint'],
          'value_numeric':numeric,
          'unit':r['explicit_unit'] if r['explicit_unit'] else '',
          'unit_evidence':'EXPLICIT_IN_COLUMN_NAME' if r['explicit_unit'] else 'SOURCE_UNIT_NOT_EXPLICIT',
          'acquisition_identifier':'',
          'model_version':'',
          'provenance_identifier':'',
          'diagnostic_status':''
        })

long_df=pd.DataFrame(records)
if long_df.empty:
    raise RuntimeError('No numeric longitudinal evidence rows were produced.')

# Deterministic cardinality check.
keycols=['person_key_private','anatomy','laterality','timepoint','source_column_token']
duplicate_rows=int(long_df.duplicated(keycols).sum())

PRIVATE_LONG=PRIVATE_RUN/'ICHI2027_Phase52_NORMALIZED_SOURCE_PRIVATE.parquet'
try:
    long_df.to_parquet(PRIVATE_LONG,index=False)
except Exception:
    PRIVATE_LONG=PRIVATE_RUN/'ICHI2027_Phase52_NORMALIZED_SOURCE_PRIVATE.csv.gz'
    long_df.to_csv(PRIVATE_LONG,index=False,compression='gzip')

# Readback integrity.
if PRIVATE_LONG.suffix=='.parquet':
    rb=pd.read_parquet(PRIVATE_LONG)
else:
    rb=pd.read_csv(PRIVATE_LONG)
assert len(rb)==len(long_df)

print('PASS: private normalized source evidence frozen.')
print('Normalized rows:',len(long_df))
print('Duplicate normalized keys:',duplicate_rows)


PASS: private normalized source evidence frozen.
Normalized rows: 4128
Duplicate normalized keys: 0


In [7]:
# CODE CELL 7/9 — Aggregate real-source eligibility and feature coverage

persons=set(long_df['person_key_private'])

def complete_persons(anatomy_name,sides=('L','R'),tps=('BASELINE','M12')):
    req={(anatomy_name,s,tp) for s in sides for tp in tps}
    out=set()
    for pid,g in long_df[long_df['anatomy']==anatomy_name].groupby('person_key_private'):
        have=set(zip(g['anatomy'],g['laterality'],g['timepoint']))
        if req<=have: out.add(pid)
    return out

hip_complete=complete_persons('hippocampus')
ent_complete=complete_persons('entorhinal')
mid_complete=complete_persons('middle_temporal')

coverage_rows=[]
for (a,side,tp),g in long_df.groupby(['anatomy','laterality','timepoint'],dropna=False):
    coverage_rows.append({
      'anatomy':a,
      'laterality':side,
      'timepoint':tp,
      'measurement_rows':len(g),
      'unique_persons':g['person_key_private'].nunique(),
      'numeric_nonmissing_rows':int(g['value_numeric'].notna().sum()),
      'explicit_unit_rows':int((g['unit_evidence']=='EXPLICIT_IN_COLUMN_NAME').sum())
    })

eligibility=[{
 'source_file_token':SELECTED_SOURCE_TOKEN,
 'source_rows':len(df),
 'source_unique_persons':len(persons),
 'normalized_longitudinal_measurement_rows':len(long_df),
 'persons_complete_bilateral_hippocampus_baseline_m12':len(hip_complete),
 'persons_complete_bilateral_entorhinal_baseline_m12':len(ent_complete),
 'persons_complete_bilateral_middle_temporal_baseline_m12':len(mid_complete),
 'persons_complete_all_three_bilateral_regions_baseline_m12':len(hip_complete & ent_complete & mid_complete),
 'persons_with_source_mci_status':0,
 'clinical_mci_status_available':False,
 'explicit_unit_rows':int((long_df['unit_evidence']=='EXPLICIT_IN_COLUMN_NAME').sum()),
 'acquisition_identifier_rows':0,
 'model_version_rows':0,
 'provenance_identifier_rows':0,
 'duplicate_normalized_keys':duplicate_rows,
 'source_ids_exported':False
}]

print('Aggregate authorized source evidence:')
for k,v in eligibility[0].items():
    if k!='source_file_token': print(k,':',v)


Aggregate authorized source evidence:
source_rows : 258
source_unique_persons : 258
normalized_longitudinal_measurement_rows : 4128
persons_complete_bilateral_hippocampus_baseline_m12 : 258
persons_complete_bilateral_entorhinal_baseline_m12 : 258
persons_complete_bilateral_middle_temporal_baseline_m12 : 0
persons_complete_all_three_bilateral_regions_baseline_m12 : 0
persons_with_source_mci_status : 0
clinical_mci_status_available : False
explicit_unit_rows : 0
acquisition_identifier_rows : 0
model_version_rows : 0
provenance_identifier_rows : 0
duplicate_normalized_keys : 0
source_ids_exported : False


In [8]:
# CODE CELL 8/9 — Real-source gaps and next transformation gate

gaps=[
 {
  'dimension':'authorization',
  'status':'AVAILABLE',
  'evidence':'Private local attestation validated',
  'blocks_fhir_source_representation':False,
  'blocks_official_armA':False
 },
 {
  'dimension':'longitudinal_bilateral_hippocampus',
  'status':'AVAILABLE' if len(hip_complete)>0 else 'ABSENT',
  'evidence':f'{len(hip_complete)} persons complete at baseline and M12',
  'blocks_fhir_source_representation':len(hip_complete)==0,
  'blocks_official_armA':len(hip_complete)==0
 },
 {
  'dimension':'explicit_measurement_unit',
  'status':'AVAILABLE' if eligibility[0]['explicit_unit_rows']>0 else 'SOURCE_NOT_EXPLICIT',
  'evidence':f"{eligibility[0]['explicit_unit_rows']} normalized rows have explicit unit evidence",
  'blocks_fhir_source_representation':False,
  'blocks_official_armA':eligibility[0]['explicit_unit_rows']==0
 },
 {
  'dimension':'acquisition_or_dicom_identifier',
  'status':'SOURCE_NOT_PRESENT',
  'evidence':'No acquisition/study/series/image identifier was established by Phase51',
  'blocks_fhir_source_representation':False,
  'blocks_official_armA':True
 },
 {
  'dimension':'extraction_model_version',
  'status':'SOURCE_NOT_PRESENT',
  'evidence':'No model/software-version field was established by Phase51',
  'blocks_fhir_source_representation':False,
  'blocks_official_armA':True
 },
 {
  'dimension':'clinical_mci_status',
  'status':'SOURCE_NOT_PRESENT',
  'evidence':'No diagnosis/group field was established by Phase51',
  'blocks_fhir_source_representation':False,
  'blocks_official_armA':True
 }
]

ready_for_fhir_source=(len(hip_complete)>0)
ready_for_official_armA=ready_for_fhir_source and not any(g['blocks_official_armA'] for g in gaps)

if ready_for_fhir_source:
    next_step='PHASE53_BUILD_PRIVATE_FHIR_R4_SOURCE_EVIDENCE_WITH_EXPLICIT_MISSINGNESS'
else:
    next_step='BLOCKED_NO_COMPLETE_BILATERAL_HIPPOCAMPUS_BASELINE_M12_PERSONS'

print('Ready for private FHIR source representation:',ready_for_fhir_source)
print('Ready for official Arm A:',ready_for_official_armA)
print('Next step:',next_step)


Ready for private FHIR source representation: True
Ready for official Arm A: False
Next step: PHASE53_BUILD_PRIVATE_FHIR_R4_SOURCE_EVIDENCE_WITH_EXPLICIT_MISSINGNESS


In [9]:
# CODE CELL 9/9 — Public-safe outputs and integrity verification

def save(name,rows,fields=None):
    p=STAGE/name
    write_csv(p,rows,fields)
    return p

public_mapping=[]
for r in mapped:
    public_mapping.append({
      'source_column_token':r['column_token'],
      'anatomy':r['anatomy'],
      'laterality':r['laterality'],
      'timepoint':r['timepoint'],
      'explicit_unit':r['explicit_unit'],
      'original_column_name_exported':False
    })

artifacts=[]
artifacts.append(save('ICHI2027_Phase52_PUBLIC_COLUMN_MAPPING.csv',public_mapping))
artifacts.append(save('ICHI2027_Phase52_FEATURE_COVERAGE.csv',coverage_rows))
artifacts.append(save('ICHI2027_Phase52_AGGREGATE_ELIGIBILITY.csv',eligibility))
artifacts.append(save('ICHI2027_Phase52_SOURCE_GAPS.csv',gaps))

manifest_public={
 'project':PROJECT,
 'phase':PHASE,
 'source_file_token':SELECTED_SOURCE_TOKEN,
 'source_bytes':source_stat.st_size,
 'source_rows':len(df),
 'source_columns':len(cols),
 'normalized_measurement_rows':len(long_df),
 'private_manifest_created':True,
 'private_normalized_source_created':True,
 'authorization_attestation_valid':True,
 'source_sha256_publicly_exported':False,
 'private_path_exported':False,
 'source_ids_exported':False,
 'created_utc':dt.datetime.now(dt.timezone.utc).isoformat()
}
manifest_path=STAGE/'ICHI2027_Phase52_SOURCE_MANIFEST_PUBLIC.json'
manifest_path.write_text(json.dumps(manifest_public,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(manifest_path)

readme=STAGE/'ICHI2027_Phase52_README.txt'
readme.write_text(
 'Phase52 freezes an authorized real-source reference for the later source-to-target fidelity experiment.\n'
 'The exact source path, exact source SHA-256, original column names, person identifiers, and normalized participant-level evidence remain private.\n'
 'No measurement unit, acquisition/DICOM identifier, model version, provenance identifier, or clinical diagnosis is invented when the selected source does not provide it.\n'
 'Phase52 does not execute FHIR-to-OMOP transformation, MI-CDM loading, or the Transformation Fidelity Layer.\n',
 encoding='utf-8'
)
artifacts.append(readme)

summary={
 'project':PROJECT,
 'phase':PHASE,
 'run_utc':dt.datetime.now(dt.timezone.utc).isoformat(),
 'input_phase51_shareable_sha256':sha256(P51),
 'selected_source_token':SELECTED_SOURCE_TOKEN,
 'authorization_attestation_valid':True,
 'private_source_manifest_created':True,
 'private_normalized_source_created':True,
 'source_rows':len(df),
 'source_unique_persons':len(persons),
 'normalized_longitudinal_measurement_rows':len(long_df),
 'persons_complete_bilateral_hippocampus_baseline_m12':len(hip_complete),
 'persons_complete_bilateral_entorhinal_baseline_m12':len(ent_complete),
 'persons_complete_bilateral_middle_temporal_baseline_m12':len(mid_complete),
 'persons_complete_all_three_regions_baseline_m12':len(hip_complete & ent_complete & mid_complete),
 'explicit_unit_rows':eligibility[0]['explicit_unit_rows'],
 'clinical_mci_status_available':False,
 'acquisition_identifier_available':False,
 'model_version_available':False,
 'provenance_identifier_available':False,
 'ready_for_private_fhir_source_representation':ready_for_fhir_source,
 'ready_for_official_armA':ready_for_official_armA,
 'next_step':next_step,
 'real_authorized_source_normalized':True,
 'real_authorized_mri_processed_by_proposed_fidelity_method':False,
 'official_omop_etl_executed':False,
 'official_micdm_load_executed':False,
 'independently_blinded_external_validation':False,
 'privacy':'PUBLIC_AGGREGATES_ONLY_PRIVATE_PATH_SHA_COLUMNS_IDS_AND_VALUES_RETAINED_PRIVATELY',
 'artifact_sha256':{p.name:sha256(p) for p in artifacts},
 'warning':'Real-source normalization establishes an immutable authorized source reference only. It does not establish clinical MCI eligibility, source units, DICOM provenance, official target transformation, or real-data fidelity performance.'
}

summary_path=STAGE/'ICHI2027_Phase52_SHAREABLE.json'
summary_path.write_text(json.dumps(summary,sort_keys=True,indent=2)+'\n',encoding='utf-8')
artifacts.append(summary_path)

zip_path=STAGE/'ICHI2027_Phase52_PUBLIC_OUTPUTS.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in artifacts:z.write(p,p.name)

OUT.mkdir(parents=True,exist_ok=False)
for p in artifacts+[zip_path]:
    dest=OUT/p.name
    for attempt in range(3):
        try:
            shutil.copyfile(p,dest)
            assert sha256(dest)==sha256(p)
            break
        except Exception as e:
            if attempt==2:raise RuntimeError('Drive save/readback verification failed for '+p.name) from e
            time.sleep(1+attempt)

print('SUCCESS: Phase52 authorized source reference frozen.')
print('Complete bilateral hippocampus baseline/M12 persons:',len(hip_complete))
print('Ready for private FHIR source representation:',ready_for_fhir_source)
print('Ready for official Arm A:',ready_for_official_armA)
print('Next step:',next_step)
print('Shareable:',OUT/summary_path.name)
print('Public ZIP:',OUT/zip_path.name)


SUCCESS: Phase52 authorized source reference frozen.
Complete bilateral hippocampus baseline/M12 persons: 258
Ready for private FHIR source representation: True
Ready for official Arm A: False
Next step: PHASE53_BUILD_PRIVATE_FHIR_R4_SOURCE_EVIDENCE_WITH_EXPLICIT_MISSINGNESS
Shareable: /content/drive/MyDrive/ICHI2027/Phase52/run_20261001T051828_306695Z/ICHI2027_Phase52_SHAREABLE.json
Public ZIP: /content/drive/MyDrive/ICHI2027/Phase52/run_20261001T051828_306695Z/ICHI2027_Phase52_PUBLIC_OUTPUTS.zip


## What Phase 52 unlocks

If the selected authorized source contains complete bilateral hippocampal baseline/M12 evidence for at least one person, Phase 53 can construct a **private FHIR R4 source representation** from this frozen normalized source.

Missing source information must remain explicit:

- no unit → no fabricated `mm3`;
- no DICOM/study/series identifier → record source-link absence;
- no model version → record model-lineage absence;
- no diagnosis field → do not call the cohort clinically attested MCI.

Only after a target transformation exists can the frozen neuroimaging fidelity method be applied to real source→target events.
